# 06 — SHAP explainability + directional sanity checks

Explains the q50 valuation model (notebook 05) with **SHAP** — for each prediction, which features
pushed the price up or down, and by how much. Also runs *directional* checks: economically the model
must learn that **more mileage → lower price** and **older → lower price**. If SHAP shows the opposite,
the model learned nonsense and we'd catch it here.

In [1]:
import json, joblib
import numpy as np, pandas as pd, shap
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt

bundle = joblib.load("../backend-api/models/valuation_model.joblib")
df = pd.read_parquet("../data/processed/listings_clean.parquet")
CAT, NUM, FEATURES = bundle["categorical_features"], bundle["numeric_features"], bundle["features"]

X = df[FEATURES].copy()
for c in CAT: X[c] = df[c].astype(str).map(bundle["cat_maps"][c]).astype("int32")
X[NUM] = X[NUM].astype("float32")

model = bundle["models"]["q50"]
expl = shap.TreeExplainer(model)
sv = expl.shap_values(X)
print("SHAP matrix:", sv.shape)

SHAP matrix: (672, 13)


### Global feature importance (mean |SHAP|)

In [2]:
imp = pd.Series(np.abs(sv).mean(0), index=FEATURES).sort_values(ascending=False)
print(imp.round(4))
shap.summary_plot(sv, X, feature_names=FEATURES, show=False, plot_size=(8,5))
plt.tight_layout(); plt.savefig("../docs/shap_summary.png", dpi=120, bbox_inches="tight"); plt.close()
print("saved ../docs/shap_summary.png")

noOfCylinders       0.3631
year                0.3076
kilometers          0.2072
make                0.1901
model               0.0814
bodyType            0.0762
mileage_per_year    0.0292
age                 0.0207
sellerType          0.0158
regionalSpecs       0.0138
fuelType            0.0135
city                0.0038
transmissionType    0.0014
dtype: float32


saved ../docs/shap_summary.png


### Directional sanity — the honest 'did it learn real economics' check

In [3]:
checks = {}
for feat in ["kilometers","age"]:
    j = FEATURES.index(feat)
    # correlation between the feature value and its SHAP contribution
    r = float(np.corrcoef(X[feat].values, sv[:,j])[0,1])
    checks[feat] = {"shap_corr": round(r,3), "expected": "negative", "pass": bool(r < 0)}
for feat in ["year"]:
    j = FEATURES.index(feat)
    r = float(np.corrcoef(X[feat].values, sv[:,j])[0,1])
    checks[feat] = {"shap_corr": round(r,3), "expected": "positive", "pass": bool(r > 0)}
print(json.dumps(checks, indent=2))
assert checks["kilometers"]["pass"] and checks["age"]["pass"], "Model learned wrong mileage/age direction!"
print("\nDirectional checks PASSED — mileage & age depress price, newer year lifts it.")
json.dump({"global_importance": imp.round(4).to_dict(), "directional_checks": checks},
          open("../eval/shap_report.json","w"), indent=2)
print("saved ../eval/shap_report.json")

{
  "kilometers": {
    "shap_corr": -0.861,
    "expected": "negative",
    "pass": true
  },
  "age": {
    "shap_corr": -0.896,
    "expected": "negative",
    "pass": true
  },
  "year": {
    "shap_corr": 0.918,
    "expected": "positive",
    "pass": true
  }
}

Directional checks PASSED — mileage & age depress price, newer year lifts it.
saved ../eval/shap_report.json


### Takeaway
The SHAP report is the model's audit trail: it shows the valuation is driven by economically sensible
factors, not spurious correlations, and the directional assertion fails loudly if that ever breaks.